# Reto Semana 3: Analizador de Ventas

## Programacion para Ciencia de Datos
**Instituto Politecnico Nacional** | Semestre Febrero-Julio 2026

---

## El Escenario del Mundo Real

Eres analista de datos en una tienda de tecnologia. Cada dia se generan cientos de transacciones de ventas. Tu jefe necesita un **reporte consolidado por producto** que muestre:

- Cuantas unidades se vendieron de cada producto
- Cual fue el ingreso total por producto
- Cual fue el precio promedio de venta

El reporte debe estar **ordenado por ingreso total** (los productos mas rentables primero).

```
╔════════════════════════════════════════════════════════════════════════════╗
║                      ANALIZADOR DE VENTAS                                  ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║   TRANSACCIONES INDIVIDUALES          REPORTE CONSOLIDADO                  ║
║   ──────────────────────────          ────────────────────                 ║
║                                                                            ║
║   2026-01-01,Laptop,2,15000     ┌───────────────────────────────────┐      ║
║   2026-01-02,Mouse,10,250   ──> │ producto | unidades | ingreso     │      ║
║   2026-01-03,Laptop,1,14500     │ ─────────┼──────────┼──────────── │      ║
║   2026-01-04,Teclado,5,800      │ Laptop   │    3     │ 44500.00    │      ║
║   2026-01-05,Mouse,8,250        │ Mouse    │   18     │  4500.00    │      ║
║                                 │ Teclado  │    5     │  4000.00    │      ║
║                                 └───────────────────────────────────┘      ║
║                                                                            ║
╚════════════════════════════════════════════════════════════════════════════╝
```

Este reto te ensenara a **agrupar datos** usando diccionarios, una habilidad fundamental en ciencia de datos.

---

## Especificacion de Entrada

### Formato
- Archivo **CSV** leido desde **stdin**
- **Primera linea**: Encabezados: `fecha,producto,cantidad,precio_unitario`
- **Siguientes lineas**: Una transaccion por linea

### Columnas

| Columna | Tipo | Descripcion | Ejemplo |
|---------|------|-------------|----------|
| `fecha` | texto | Fecha de la venta (YYYY-MM-DD) | `2026-01-15` |
| `producto` | texto | Nombre del producto | `Laptop`, `Mouse` |
| `cantidad` | entero | Unidades vendidas | `2`, `10` |
| `precio_unitario` | decimal | Precio por unidad | `15000.00`, `250.50` |

### Ejemplo de Entrada
```
fecha,producto,cantidad,precio_unitario
2026-01-01,Laptop,2,15000.00
2026-01-02,Mouse,10,250.00
2026-01-03,Laptop,1,14500.00
2026-01-04,Teclado,5,800.00
2026-01-05,Mouse,8,250.00
```

---

## Reglas de Procesamiento

### Regla 1: Agrupar por Producto

Todas las transacciones del **mismo producto** deben consolidarse en una sola fila del reporte.

```
Transacciones de "Laptop":
  - 2026-01-01: 2 unidades @ $15,000
  - 2026-01-03: 1 unidad @ $14,500
  
Consolidado "Laptop":
  - Total unidades: 2 + 1 = 3
  - Ingreso total: (2 × 15000) + (1 × 14500) = 44,500
```

### Regla 2: Calcular Metricas por Producto

Para cada producto calcular:

| Metrica | Formula | Ejemplo (Laptop) |
|---------|---------|------------------|
| `unidades_vendidas` | Suma de todas las cantidades | 2 + 1 = **3** |
| `ingreso_total` | Suma de (cantidad × precio) de cada transaccion | (2×15000) + (1×14500) = **44500.00** |
| `precio_promedio` | ingreso_total / unidades_vendidas | 44500 / 3 = **14833.33** |

In [ ]:
# Ejemplo de los calculos

# Transacciones de Laptop
transacciones_laptop = [
    {"cantidad": 2, "precio": 15000.00},
    {"cantidad": 1, "precio": 14500.00}
]

# Calcular metricas
unidades = sum(t["cantidad"] for t in transacciones_laptop)
ingreso = sum(t["cantidad"] * t["precio"] for t in transacciones_laptop)
promedio = ingreso / unidades

print(f"Laptop:")
print(f"  Unidades vendidas: {unidades}")
print(f"  Ingreso total: ${ingreso:,.2f}")
print(f"  Precio promedio: ${promedio:,.2f}")

### Regla 3: Ordenar por Ingreso Total (Descendente)

El reporte debe estar ordenado de **mayor a menor ingreso total**.

```
Orden CORRECTO (descendente por ingreso):
  1. Laptop:  $44,500.00  <- Mayor ingreso primero
  2. Mouse:   $4,500.00
  3. Teclado: $4,000.00   <- Menor ingreso al final
```

### Regla 4: Formato de Numeros

- **Valores monetarios** (ingreso_total, precio_promedio): **2 decimales**
- **Unidades**: entero (sin decimales)

### Regla 5: Ignorar Lineas Invalidas

Si una linea tiene:
- Cantidad no numerica
- Precio no numerico
- Menos de 4 columnas

**Ignorar esa linea** (no incluir en calculos ni salida).

---

## Especificacion de Salida

### Formato
- Archivo **CSV** escrito a **stdout**
- Primera linea: encabezados
- Una linea por producto, ordenado por ingreso descendente

### Columnas de Salida

| Columna | Tipo | Formato |
|---------|------|--------|
| `producto` | texto | Nombre del producto |
| `unidades_vendidas` | entero | Sin decimales |
| `ingreso_total` | decimal | 2 decimales |
| `precio_promedio` | decimal | 2 decimales |

### Ejemplo Completo

**Entrada:**
```
fecha,producto,cantidad,precio_unitario
2026-01-01,Laptop,2,15000.00
2026-01-02,Mouse,10,250.00
2026-01-03,Laptop,1,14500.00
2026-01-04,Teclado,5,800.00
2026-01-05,Mouse,8,250.00
```

**Salida:**
```
producto,unidades_vendidas,ingreso_total,precio_promedio
Laptop,3,44500.00,14833.33
Mouse,18,4500.00,250.00
Teclado,5,4000.00,800.00
```

---

## Estrategia de Solucion: Usar Diccionarios para Agrupar

La clave de este reto es usar un **diccionario** donde:
- La **clave** es el nombre del producto
- El **valor** es otro diccionario con los totales acumulados

```python
# Estructura del diccionario de agrupacion
productos = {
    "Laptop": {
        "unidades": 3,
        "ingreso": 44500.00
    },
    "Mouse": {
        "unidades": 18,
        "ingreso": 4500.00
    },
    "Teclado": {
        "unidades": 5,
        "ingreso": 4000.00
    }
}
```

---

## Guia de Implementacion Paso a Paso

### Paso 1: Leer y Parsear el CSV

In [ ]:
# Simulacion de datos de entrada
datos_entrada = """fecha,producto,cantidad,precio_unitario
2026-01-01,Laptop,2,15000.00
2026-01-02,Mouse,10,250.00
2026-01-03,Laptop,1,14500.00
2026-01-04,Teclado,5,800.00
2026-01-05,Mouse,8,250.00"""

lineas = datos_entrada.strip().split('\n')

# Saltar encabezado
for linea in lineas[1:]:  # Desde la segunda linea
    partes = linea.split(',')
    print(f"Partes: {partes}")
    
    if len(partes) == 4:
        fecha = partes[0]
        producto = partes[1]
        cantidad = int(partes[2])
        precio = float(partes[3])
        print(f"  -> {producto}: {cantidad} x ${precio}")

### Paso 2: Agrupar con Diccionario

In [ ]:
# Diccionario para acumular
productos = {}

# Simulacion de transacciones
transacciones = [
    ("Laptop", 2, 15000.00),
    ("Mouse", 10, 250.00),
    ("Laptop", 1, 14500.00),
    ("Teclado", 5, 800.00),
    ("Mouse", 8, 250.00)
]

for producto, cantidad, precio in transacciones:
    # Si el producto no existe, crearlo
    if producto not in productos:
        productos[producto] = {
            "unidades": 0,
            "ingreso": 0.0
        }
    
    # Acumular valores
    productos[producto]["unidades"] += cantidad
    productos[producto]["ingreso"] += cantidad * precio

print("Diccionario agrupado:")
for prod, datos in productos.items():
    print(f"  {prod}: {datos}")

### Paso 3: Calcular Precio Promedio

In [ ]:
# Agregar precio promedio a cada producto
for producto in productos:
    unidades = productos[producto]["unidades"]
    ingreso = productos[producto]["ingreso"]
    productos[producto]["promedio"] = ingreso / unidades

print("Con precio promedio:")
for prod, datos in productos.items():
    print(f"  {prod}:")
    print(f"    Unidades: {datos['unidades']}")
    print(f"    Ingreso: ${datos['ingreso']:,.2f}")
    print(f"    Promedio: ${datos['promedio']:,.2f}")

### Paso 4: Ordenar por Ingreso Descendente

In [ ]:
# Convertir a lista de tuplas para ordenar
# Cada tupla: (nombre_producto, diccionario_de_datos)
lista_productos = list(productos.items())
print("Como lista:")
for item in lista_productos:
    print(f"  {item}")

# Ordenar por ingreso (descendente)
# key=lambda x: x[1]["ingreso"] -> ordena por el valor de "ingreso" en el diccionario
lista_ordenada = sorted(lista_productos, key=lambda x: x[1]["ingreso"], reverse=True)

print("\nOrdenada por ingreso (desc):")
for nombre, datos in lista_ordenada:
    print(f"  {nombre}: ${datos['ingreso']:,.2f}")

### Paso 5: Generar Salida CSV

In [ ]:
# Imprimir encabezado
print("producto,unidades_vendidas,ingreso_total,precio_promedio")

# Imprimir cada producto
for nombre, datos in lista_ordenada:
    unidades = datos["unidades"]
    ingreso = datos["ingreso"]
    promedio = datos["promedio"]
    print(f"{nombre},{unidades},{ingreso:.2f},{promedio:.2f}")

---

## Codigo Completo de Referencia

In [ ]:
# ESTRUCTURA COMPLETA DEL PROGRAMA

codigo_completo = '''
import sys

def main():
    # Diccionario para agrupar por producto
    productos = {}
    
    primera_linea = True
    
    # Leer todas las lineas
    for linea in sys.stdin:
        linea = linea.strip()
        
        # Saltar encabezado
        if primera_linea:
            primera_linea = False
            continue
        
        # Saltar lineas vacias
        if not linea:
            continue
        
        # Parsear linea
        partes = linea.split(',')
        if len(partes) != 4:
            continue  # Ignorar lineas invalidas
        
        fecha = partes[0]
        producto = partes[1]
        
        # Convertir cantidad y precio (con manejo de errores)
        try:
            cantidad = int(partes[2])
            precio = float(partes[3])
        except ValueError:
            continue  # Ignorar si no son numeros validos
        
        # Crear entrada si no existe
        if producto not in productos:
            productos[producto] = {
                "unidades": 0,
                "ingreso": 0.0
            }
        
        # Acumular
        productos[producto]["unidades"] += cantidad
        productos[producto]["ingreso"] += cantidad * precio
    
    # Calcular precio promedio
    for prod in productos:
        unidades = productos[prod]["unidades"]
        ingreso = productos[prod]["ingreso"]
        productos[prod]["promedio"] = ingreso / unidades if unidades > 0 else 0
    
    # Ordenar por ingreso descendente
    productos_ordenados = sorted(
        productos.items(),
        key=lambda x: x[1]["ingreso"],
        reverse=True
    )
    
    # Imprimir salida
    print("producto,unidades_vendidas,ingreso_total,precio_promedio")
    for nombre, datos in productos_ordenados:
        print(f"{nombre},{datos[\'unidades\']},{datos[\'ingreso\']:.2f},{datos[\'promedio\']:.2f}")

if __name__ == "__main__":
    main()
'''

print(codigo_completo)

---

## Tabla de Casos de Prueba

### Caso 1: Ejemplo Principal

**Entrada:**
```
fecha,producto,cantidad,precio_unitario
2026-01-01,Laptop,2,15000.00
2026-01-02,Mouse,10,250.00
2026-01-03,Laptop,1,14500.00
2026-01-04,Teclado,5,800.00
2026-01-05,Mouse,8,250.00
```

**Salida:**
```
producto,unidades_vendidas,ingreso_total,precio_promedio
Laptop,3,44500.00,14833.33
Mouse,18,4500.00,250.00
Teclado,5,4000.00,800.00
```

### Caso 2: Con Datos Invalidos

**Entrada:**
```
fecha,producto,cantidad,precio_unitario
2026-01-01,Laptop,2,15000.00
2026-01-02,Mouse,abc,250.00
2026-01-03,Teclado,5,invalid
2026-01-04,Laptop,1,14500.00
linea,incompleta
```

**Salida:**
```
producto,unidades_vendidas,ingreso_total,precio_promedio
Laptop,3,44500.00,14833.33
```

### Caso 3: Producto Unico

**Entrada:**
```
fecha,producto,cantidad,precio_unitario
2026-01-01,Monitor,1,5000.00
```

**Salida:**
```
producto,unidades_vendidas,ingreso_total,precio_promedio
Monitor,1,5000.00,5000.00
```

---

## Diagrama de Flujo

```
                    ┌─────────────────────┐
                    │      INICIO         │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ productos = {}      │
                    └──────────┬──────────┘
                               │
              ┌────────────────▼───────────────┐
              │   Para cada linea en stdin     │<────────────┐
              └────────────────┬───────────────┘             │
                               │                             │
                               ▼                             │
                    ┌─────────────────────┐     NO           │
                    │ Linea valida?       │──────────────────┤
                    └──────────┬──────────┘                  │
                               │ SI                          │
                               ▼                             │
                    ┌─────────────────────┐                  │
                    │ Extraer:            │                  │
                    │ producto, cantidad, │                  │
                    │ precio              │                  │
                    └──────────┬──────────┘                  │
                               │                             │
                               ▼                             │
                    ┌─────────────────────┐                  │
                    │ producto existe     │                  │
                    │ en diccionario?     │                  │
                    └──────────┬──────────┘                  │
                          NO   │   SI                        │
                    ┌──────────┴──────────┐                  │
                    ▼                     ▼                  │
             ┌─────────────┐    ┌─────────────────┐          │
             │ Crear nueva │    │ Acumular:       │          │
             │ entrada     │    │ unidades +=     │          │
             └──────┬──────┘    │ ingreso +=      │          │
                    │           └────────┬────────┘          │
                    └────────────────────┴───────────────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ Calcular promedios  │
                    │ para cada producto  │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ Ordenar por ingreso │
                    │ descendente         │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ Imprimir CSV        │
                    │ de salida           │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │        FIN          │
                    └─────────────────────┘
```

---

## Conceptos Clave Aplicados

Este reto integra varios conceptos de la semana:

| Concepto | Uso en el Reto |
|----------|----------------|
| **Diccionarios** | Agrupar transacciones por producto |
| **Diccionarios anidados** | Almacenar unidades e ingreso por producto |
| **Iteracion sobre dict** | Procesar cada producto al final |
| **Listas** | Convertir dict a lista para ordenar |
| **sorted() con key** | Ordenar por ingreso descendente |
| **Lambda** | Funcion de ordenamiento |
| **f-strings con formato** | Imprimir con 2 decimales |
| **Manejo de excepciones** | try/except para datos invalidos |

---

## Errores Comunes

### 1. No inicializar el producto antes de acumular

```python
# INCORRECTO - KeyError si el producto no existe
productos[producto]["unidades"] += cantidad

# CORRECTO - Verificar primero
if producto not in productos:
    productos[producto] = {"unidades": 0, "ingreso": 0.0}
productos[producto]["unidades"] += cantidad
```

### 2. Division por cero en promedio

```python
# INCORRECTO - Puede dar ZeroDivisionError
promedio = ingreso / unidades

# CORRECTO - Verificar
promedio = ingreso / unidades if unidades > 0 else 0
```

### 3. Ordenar el diccionario directamente

```python
# INCORRECTO - No puedes ordenar un dict directamente
productos_ordenados = sorted(productos)

# CORRECTO - Ordenar los items
productos_ordenados = sorted(productos.items(), key=lambda x: x[1]["ingreso"], reverse=True)
```

### 4. Olvidar el orden descendente

```python
# INCORRECTO - Ordena ascendente por defecto
sorted(productos.items(), key=lambda x: x[1]["ingreso"])

# CORRECTO - Agregar reverse=True
sorted(productos.items(), key=lambda x: x[1]["ingreso"], reverse=True)
```

### 5. Formato incorrecto de decimales

```python
# INCORRECTO
print(f"{ingreso}")
# Salida: 44500.0

# CORRECTO
print(f"{ingreso:.2f}")
# Salida: 44500.00
```

---

## Archivo de Prueba Completo

Crea `entrada_prueba.txt`:

In [ ]:
entrada_prueba = """fecha,producto,cantidad,precio_unitario
2026-01-01,Laptop,2,15000.00
2026-01-02,Mouse,10,250.00
2026-01-03,Laptop,1,14500.00
2026-01-04,Teclado,5,800.00
2026-01-05,Mouse,8,250.00
2026-01-06,Monitor,3,6000.00
2026-01-07,Laptop,2,15500.00
2026-01-08,Audifonos,20,350.00
2026-01-09,Mouse,5,275.00
2026-01-10,Monitor,2,5800.00"""

salida_esperada = """producto,unidades_vendidas,ingreso_total,precio_promedio
Laptop,5,75500.00,15100.00
Monitor,5,29600.00,5920.00
Audifonos,20,7000.00,350.00
Mouse,23,6875.00,298.91
Teclado,5,4000.00,800.00"""

print("=== ENTRADA ===")
print(entrada_prueba)
print("\n=== SALIDA ESPERADA ===")
print(salida_esperada)

---

## Entregables

### Repositorio en GitHub

```
reto-semana-03/
├── README.md
├── main.py
├── .gitignore
└── tests/
    ├── entrada1.txt
    ├── salida1.txt
    └── ...
```

### Criterios de Evaluacion

| Criterio | Puntos |
|----------|--------|
| Agrupacion correcta por producto | 25 |
| Calculos correctos (unidades, ingreso, promedio) | 25 |
| Ordenamiento descendente por ingreso | 15 |
| Formato de salida exacto | 15 |
| Manejo de datos invalidos | 10 |
| Codigo organizado con funciones | 5 |
| Git (commits, README) | 5 |
| **Total** | **100** |

---

## Fecha de Entrega

**Viernes de la Semana 3, 23:59 hrs**

---

*Reto Semana 3 - Programacion para Ciencia de Datos - IPN 2026*